# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** *line item from an order.*

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)
display(lines)

assert len(lines) == 4, f"Expected 4 rows, got {len(lines)}"
assert lines['qty'].sum() == 7, f"Expected sum 7, got {lines['qty'].sum()}"

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors_df = pd.DataFrame(vendors_json)

joined = pd.merge(lines, vendors_df, on='vendor_id', how='left', indicator=True)
display(joined)

unmatched = joined[joined['_merge'] == 'left_only']
if not unmatched.empty:
    unmatched_vendor = unmatched['vendor_id'].iloc[0]
    unmatched_qty = unmatched['qty'].sum()
    print(f"Vendor {unmatched_vendor} did not match and has {unmatched_qty} units attached.")
else:
    print("All vendors matched.")

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


All vendors matched.


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# We use dropna=False in groupby so that any unmatched vendor (name is NaN)
# appears in the result. If we dropped them, we might undercount the total sold items.

vendor_totals = joined.groupby('name', dropna=False)['qty'].sum().reset_index()
display(vendor_totals)

,name,qty
0,Cav Merch,1
1,Hoos Burgers,6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_totals = joined.groupby('zone')['qty'].sum()
display(zone_totals)

best_zone = zone_totals.idxmax()
zoned_total = zone_totals.sum()
total_qty = joined['qty'].sum()
difference = total_qty - zoned_total

print(f"Zone {best_zone} sold the most items.")
print(f"Total items with a known zone: {zoned_total}")
print(f"The difference of {difference} items belongs to vendors that did not match and therefore have no zone.")

,qty
zone,
A,7


Zone A sold the most items.
Total items with a known zone: 7
The difference of 0 items belongs to vendors that did not match and therefore have no zone.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

for record in ragged_json:
    if 'lines' not in record:
        record['lines'] = []

ragged_lines = pd.json_normalize(
    ragged_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)
display(ragged_lines)

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

Having one row per line item makes it easier to analyze individual items instead of entire orders. For example, we can easily calculate the total quantity sold for each item, which would be harder if multiple items were nested within one order.

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A missing quantity means the data was not provided, while a quantity of zero means no units were ordered. If we treat missing values as zero, we could underestimate total sales and make inaccurate conclusions because we would be treating unknown values as actual zeros.